In [ ]:
# E23 P6a: EAGLE-4 vLLM fork offline env + CPU feasibility checks (internet ON, pinned image)
import hashlib, json, os, re, shutil, subprocess, sys, tarfile, time
from pathlib import Path
T0 = time.time()
GLIBCFIX = '"""E23 P2: make the Merlin release .so files load on glibc 2.35 (Kaggle image, Ubuntu 22.04).\n\nThe Merlin 3.0.0 wheel\'s vllm/_C_stable_libtorch.abi3.so needs libc.so.6 version GLIBC_2.38, only for\n__isoc23_strtol / __isoc23_strtoll (C23 names of strtol / strtoll that GCC 13 + glibc 2.38 headers emit).\nThis tool, per ELF file:\n  1. lists every dynamic symbol bound to a libc.so.6 version above MAX_OK (2.35);\n  2. refuses (exit 2) if any such symbol is not one the shim provides (SHIM_SYMBOLS);\n  3. sets VER_FLG_WEAK on those version-need entries, so ld.so does not fail on the missing version\n     (glibc dl-version.c: a weak version that is not found is not fatal); the symbols then bind to the\n     unversioned definitions in the LD_PRELOADed shim (dl-lookup.c accepts an unversioned definition\n     from an object that has no version info).\nNo other byte changes. Standard library only.\nusage: python glibc235_fix.py [--dry-run] <file.so> ...   -> prints one GLIBCFIX {...} JSON line"""\nimport json\nimport struct\nimport sys\n\nMAX_OK = (2, 35)\nSHIM_SYMBOLS = {"__isoc23_strtol", "__isoc23_strtoll", "__isoc23_strtoul", "__isoc23_strtoull"}\nSHT_DYNSYM, SHT_GNU_VERNEED, SHT_GNU_VERSYM = 11, 0x6FFFFFFE, 0x6FFFFFFF\nVER_FLG_WEAK = 0x2\n\n\ndef glibc_ver(name):\n    if not name.startswith("GLIBC_"):\n        return None\n    try:\n        return tuple(int(x) for x in name[6:].split("."))\n    except ValueError:\n        return None\n\n\ndef cstr(buf, off):\n    return buf[off:buf.index(b"\\0", off)].decode()\n\n\ndef scan(path):\n    data = bytearray(open(path, "rb").read())\n    if data[:4] != b"\\x7fELF" or data[4] != 2 or data[5] != 1:\n        raise ValueError("not a little-endian ELF64 file")\n    e_shoff, = struct.unpack_from("<Q", data, 0x28)\n    e_shentsize, e_shnum = struct.unpack_from("<HH", data, 0x3A)\n    secs = []\n    for i in range(e_shnum):\n        sh = struct.unpack_from("<IIQQQQIIQQ", data, e_shoff + i * e_shentsize)\n        secs.append({"type": sh[1], "offset": sh[4], "size": sh[5], "link": sh[6], "entsize": sh[9]})\n    by_type = {}\n    for s in secs:\n        by_type.setdefault(s["type"], s)\n    vn = by_type.get(SHT_GNU_VERNEED)\n    rows = {"needs": [], "high": [], "high_symbols": []}\n    if vn is None:\n        return data, rows\n    strtab = secs[vn["link"]]\n    st = bytes(data[strtab["offset"]:strtab["offset"] + strtab["size"]])\n    off, high_idx = vn["offset"], {}\n    while True:\n        vn_version, vn_cnt, vn_file, vn_aux, vn_next = struct.unpack_from("<HHIII", data, off)\n        lib = cstr(st, vn_file)\n        aoff = off + vn_aux\n        for _ in range(vn_cnt):\n            vna_hash, vna_flags, vna_other, vna_name, vna_next = struct.unpack_from("<IHHII", data, aoff)\n            name = cstr(st, vna_name)\n            rows["needs"].append([lib, name, vna_flags, vna_other])\n            v = glibc_ver(name)\n            if lib == "libc.so.6" and v and v > MAX_OK:\n                rows["high"].append({"lib": lib, "version": name, "flags": vna_flags, "index": vna_other,\n                                     "flags_offset": aoff + 4})\n                high_idx[vna_other] = name\n            if not vna_next:\n                break\n            aoff += vna_next\n        if not vn_next:\n            break\n        off += vn_next\n    if high_idx:\n        dynsym, versym = by_type.get(SHT_DYNSYM), by_type.get(SHT_GNU_VERSYM)\n        dstr = secs[dynsym["link"]]\n        ds = bytes(data[dstr["offset"]:dstr["offset"] + dstr["size"]])\n        n = dynsym["size"] // dynsym["entsize"]\n        for i in range(n):\n            st_name, = struct.unpack_from("<I", data, dynsym["offset"] + i * dynsym["entsize"])\n            ndx, = struct.unpack_from("<H", data, versym["offset"] + 2 * i)\n            if (ndx & 0x7FFF) in high_idx:\n                rows["high_symbols"].append([cstr(ds, st_name), high_idx[ndx & 0x7FFF]])\n    return data, rows\n\n\ndef main(argv):\n    dry = "--dry-run" in argv\n    files = [a for a in argv if a != "--dry-run"]\n    out = {"dry_run": dry, "files": {}, "ok": True}\n    for f in files:\n        rec = {}\n        try:\n            data, rows = scan(f)\n            rec.update(high=[{k: v for k, v in h.items() if k != "flags_offset"} for h in rows["high"]],\n                       high_symbols=rows["high_symbols"])\n            unknown = sorted({s for s, _ in rows["high_symbols"]} - SHIM_SYMBOLS)\n            rec["unknown_symbols"] = unknown\n            if unknown:\n                rec["action"] = "REFUSED: symbols above glibc 2.35 that the shim does not provide"\n                out["ok"] = False\n            elif not rows["high"]:\n                rec["action"] = "none needed"\n            else:\n                for h in rows["high"]:\n                    fl, = struct.unpack_from("<H", data, h["flags_offset"])\n                    struct.pack_into("<H", data, h["flags_offset"], fl | VER_FLG_WEAK)\n                if not dry:\n                    open(f, "wb").write(data)\n                    _, again = scan(f)\n                    rec["after"] = [[h["version"], h["flags"]] for h in again["high"]]\n                    if not all(h["flags"] & VER_FLG_WEAK for h in again["high"]):\n                        rec["action"] = "FAILED: weak flag not set after write"\n                        out["ok"] = False\n                        out["files"][f] = rec\n                        continue\n                rec["action"] = "weak flag set" + (" (dry run, not written)" if dry else "")\n        except Exception as e:\n            rec["action"] = f"ERROR {type(e).__name__}: {e}"\n            out["ok"] = False\n        out["files"][f] = rec\n    print("GLIBCFIX " + json.dumps(out))\n    return 0 if out["ok"] else 2\n\n\nif __name__ == "__main__":\n    sys.exit(main(sys.argv[1:]))\n'
FLAGCHECK = '"""Parse E21\'s launch commands with the REAL vLLM argument parser of the installed version (Kaggle CPU kit; no GPU needed).\nusage: <python with vllm X> e21_flagcheck.py <flags.json> <version>   -> prints one JSON line per command, then a summary.\nflags.json = CONFIG["e21"]["flags"] of a notebook run ({"A_primary": {"vllm": "0.19.0", "flags": [...]}, ...}); only the\ncommands of <version> are checked. vLLM 0.19 commands use the api_server form (--model <path>); 0.32 commands use the\n`vllm serve <path>` form, parsed here as --model <path> with the same make_arg_parser (the serve subcommand adds only\nthe positional model tag). Each parsed namespace is also turned into AsyncEngineArgs (types, choices, JSON values)."""\nimport json\nimport sys\nimport traceback\n\nflags_all = json.load(open(sys.argv[1], encoding="utf-8"))\nversion = sys.argv[2]\nimport vllm  # noqa: E402\n\n# rev 2 (Kaggle CPU kit run 1): on this CPU-only box vLLM finds no platform, and building the argument parser already\n# fails (EngineArgs.add_cli_args -> DeviceConfig default -> "Failed to infer device type"). The parser needs only a\n# device type, so install a stub platform that says "cuda" before the parser is built (nothing here touches a GPU).\nimport vllm.platforms as _vp  # noqa: E402\ntry:\n    _vp.current_platform.device_type\n    _need_stub = not _vp.current_platform.device_type\nexcept Exception:\n    _need_stub = True\nif _need_stub:\n    from vllm.platforms.interface import UnspecifiedPlatform  # noqa: E402\n\n    class _E21StubPlatform(UnspecifiedPlatform):\n        device_name = "cuda"\n        device_type = "cuda"\n        dispatch_key = "CUDA"\n    _vp._current_platform = _E21StubPlatform()\n    out_stub = True\nelse:\n    out_stub = False\n\nout = {"vllm": vllm.__version__, "want": version, "checked": 0, "ok": 0, "errors": {}, "stub_platform": out_stub}\nassert vllm.__version__.startswith(version), (vllm.__version__, version)\ntry:\n    from vllm.utils.argparse_utils import FlexibleArgumentParser\nexcept Exception:\n    from vllm.utils import FlexibleArgumentParser\ntry:      # 0.19: entrypoints/openai/cli_args.py; 0.32 moved it to entrypoints/launchers/cli_args.py (VAL_api_D1 F7)\n    from vllm.entrypoints.openai.cli_args import make_arg_parser  # noqa: E402\nexcept ImportError:\n    from vllm.entrypoints.launchers.cli_args import make_arg_parser  # noqa: E402\nfrom vllm.engine.arg_utils import AsyncEngineArgs  # noqa: E402\n\nfor key, row in sorted(flags_all.items()):\n    if row["vllm"] != version:\n        continue\n    argv = list(row["flags"])\n    if not argv[0].startswith("--"):\n        argv = ["--model", argv[0]] + argv[1:]\n    out["checked"] += 1\n    rec = {"key": key}\n    try:\n        p = make_arg_parser(FlexibleArgumentParser())\n        ns = p.parse_args(argv)\n        # step 2 (rev 2, Kaggle CPU kit run 1): AsyncEngineArgs may need a device on this CPU-only box ("Failed to infer\n        # device type"); that one error is recorded, not a flag failure. Any other error is.\n        try:\n            ea = AsyncEngineArgs.from_cli_args(ns)\n            rec["engine_args"] = "ok"\n        except Exception as e2:\n            if "infer device type" not in str(e2):\n                raise\n            ea, rec["engine_args"] = None, "skipped on CPU: " + str(e2)[:120]\n        rec.update(ok=True, parsed={k: str(getattr(ns, k, None)) for k in (\n            "model", "served_model_name", "port", "max_num_seqs", "max_model_len", "kv_cache_dtype", "speculative_config",\n            "attention_backend", "mamba_ssm_cache_dtype", "enable_prefix_caching", "tool_call_parser", "reasoning_parser",\n            "enable_auto_tool_choice", "default_chat_template_kwargs", "language_model_only", "enable_prompt_tokens_details",\n            "generation_config", "trust_remote_code", "disable_custom_all_reduce", "max_num_batched_tokens", "disable_uvicorn_access_log",\n            "gpu_memory_utilization", "tensor_parallel_size")},\n            engine_args_values=None if ea is None else {k: str(getattr(ea, k, None)) for k in (\n                "mamba_ssm_cache_dtype", "kv_cache_dtype", "max_num_seqs", "enable_prefix_caching", "speculative_config")})\n        out["ok"] += 1\n    except SystemExit as e:\n        rec.update(ok=False, error=f"SystemExit {e.code} (argparse rejected the command)")\n    except Exception as e:\n        rec.update(ok=False, error=f"{type(e).__name__}: {e}"[:800], tb=traceback.format_exc()[-1500:])\n    if not rec["ok"]:\n        out["errors"][key] = rec.get("error")\n    print(json.dumps(rec), flush=True)\nout["all_ok"] = out["checked"] > 0 and out["ok"] == out["checked"]\nprint("FLAGCHECK", json.dumps(out), flush=True)\n'
REQS = ["eagle-harness[vllm]", "vllm==0.32.0rc1.dev283+eagle.5"]
IDX = ["https://eagle-hq.github.io/EAGLE-4/whl/cu130/", "https://download.pytorch.org/whl/cu130"]
W, T = Path("/kaggle/working"), Path("/tmp/e23p6")
PAY, LOGS = T / "eagle_offline", T / "logs"
for p in (PAY / "wheelhouse", LOGS):
    p.mkdir(parents=True, exist_ok=True)
R = {"steps": {}, "errors": [], "facts": {}}
VPY = "/tmp/eagle-off/bin/python"
STUB = T / "stub"


def sh(name, cmd, timeout=3600, ok_rc=(0,), env=None):
    t = time.time()
    with open(LOGS / f"{name}.log", "w") as f:
        r = subprocess.run(["bash", "-c", "set -o pipefail; " + cmd], stdout=f, stderr=subprocess.STDOUT, timeout=timeout,
                           env={**os.environ, **(env or {})})
    txt = (LOGS / f"{name}.log").read_text(errors="replace")
    R["steps"][name] = {"rc": r.returncode, "sec": round(time.time() - t, 1), "tail": txt[-3000:]}
    print(f"[{time.time() - T0:7.1f}s] {name}: rc {r.returncode}", flush=True)
    if r.returncode not in ok_rc:
        print(txt[-2500:], flush=True)
    return r.returncode, txt


def fail(m):
    R["errors"].append(m)
    print("ERROR:", m, flush=True)


try:
    sh("env", "python3 --version; uv --version; ldd --version | head -1; nproc; df -h /kaggle/working /tmp | cat")
    (PAY / "glibc235_fix.py").write_text(GLIBCFIX)
    (PAY / "reqs.txt").write_text("\n".join(REQS) + "\n")
    idx = " ".join(f"--extra-index-url {u}" for u in IDX)
    # 1. wheelhouse (resolved for this image's CPython 3.12 / linux x86_64, as the GPU notebook will use)
    rc, _ = sh("download", f"python3 -m pip download {' '.join(repr(x) for x in REQS)} {idx} -d {PAY}/wheelhouse "
                           f"--progress-bar off -q", timeout=5400)
    whl = sorted(p.name for p in (PAY / "wheelhouse").iterdir())
    R["facts"]["wheels"] = {"n": len(whl), "bytes": sum(p.stat().st_size for p in (PAY / "wheelhouse").iterdir()),
                            "vllm": [n for n in whl if n.startswith("vllm-")], "eagle": [n for n in whl if "eagle" in n.lower()],
                            "torch": [n for n in whl if n.startswith("torch-")], "flashinfer": [n for n in whl if "flashinfer" in n],
                            "triton": [n for n in whl if n.startswith("triton")], "sdist": [n for n in whl if not n.endswith(".whl")]}
    if rc or not R["facts"]["wheels"]["vllm"]:
        fail(f"wheelhouse download rc {rc}")
    # 2. offline install from the wheelhouse only (the GPU notebook's path): uv venv on the system CPython, no index
    (PAY / "install_eagle_offline.sh").write_text(
        "#!/usr/bin/env bash\n# usage: install_eagle_offline.sh <payload-dir> <venv-dir>  (offline: no index)\nset -euo pipefail\n"
        'PAY="$1"; V="$2"\nuv venv --python "$(command -v python3)" "$V"\n'
        'UV_OFFLINE=1 UV_PRERELEASE=allow uv pip install --python "$V/bin/python" --no-index --find-links "$PAY/wheelhouse" '
        '-r "$PAY/reqs.txt"\n'
        'echo "EAGLE_VENV=$V"\necho INSTALL OK\n')
    shutil.rmtree("/tmp/eagle-off", ignore_errors=True)
    rc, txt = sh("offline_install", f"bash {PAY}/install_eagle_offline.sh {PAY} /tmp/eagle-off", timeout=3600)
    if rc or "INSTALL OK" not in txt:
        fail("offline install failed")
    else:
        STUB.mkdir(exist_ok=True)
        if not (STUB / "libcuda.so.1").exists():
            os.symlink("/usr/local/cuda-12.8/lib64/stubs/libcuda.so", STUB / "libcuda.so.1")
        E = {"LD_LIBRARY_PATH": str(STUB), "VLLM_TARGET_DEVICE": "cuda"}
        # 3. versions + import
        rc, txt = sh("versions", f"cd /tmp && {VPY} -c \"import importlib.metadata as m, json; "
                     f"print('VERS', json.dumps({{k: m.version(k) for k in ['vllm','torch','triton','eagle-harness'] }}))\" ; "
                     f"{VPY} -m pip list 2>/dev/null | grep -i -E 'flashinfer|cutlass|cuda-|nvidia-nccl|eagle' ; "
                     f"uv pip list --python {VPY} | grep -i -E 'flashinfer|cutlass|nccl|eagle|torch|triton|vllm'", env=E, ok_rc=(0, 1))
        m = re.search(r"^VERS (.*)$", txt, re.M)
        R["facts"]["versions"] = json.loads(m.group(1)) if m else None
        rc, txt = sh("import", f"cd /tmp && {VPY} -c \"import torch, vllm, eagle_harness; print('IMPORT OK', vllm.__version__, "
                     f"torch.__version__, torch.version.cuda, eagle_harness.__file__)\"", env=E, ok_rc=(0, 1))
        R["facts"]["import_ok"] = rc == 0 and "IMPORT OK" in txt
        if not R["facts"]["import_ok"]:
            fail("import vllm / eagle_harness failed")
        # 4. CLI (v3): `vllm serve --help` cannot build its parser on a CPU box ("Failed to infer device type", v1), so the
        # EAGLE-4 launch commands are parsed with E21's flag checker (stub "cuda" platform, the fork's real parser +
        # AsyncEngineArgs), with our FP8 target and the docs' quickstart flags
        (T / "e21_flagcheck.py").write_text(FLAGCHECK)
        fp8 = "/kaggle/input/models/foysalemonshanto/qwen3-8-27b-fp8-repacked-v1/pytorch/hf-fp8/1"
        drafter = "/kaggle/input/models/foysalemonshanto/qwen3-8-27b-eagle4/transformers/default/1"
        spec = json.dumps({"method": "eagle_harness", "model": drafter, "method_config": {"mode": "static", "node_top_k": 4,
                                                                                         "tree_depth": 14, "max_verify_width": 16}})
        base = [fp8, "--tensor-parallel-size", "4", "--max-model-len", "106496", "--gpu-memory-utilization", "0.90",
                "--max-num-seqs", "16", "--max-num-batched-tokens", "4096", "--attention-backend", "CUSTOM",
                "--speculative-config", spec, "--no-enable-prefix-caching", "--reasoning-parser", "qwen3",
                "--language-model-only", "--disable-custom-all-reduce", "--served-model-name", "q27"]
        V = (R["facts"].get("versions") or {}).get("vllm") or "0.32.0rc1"
        flags = {"eagle_fp8_tp4": {"vllm": V, "flags": base},
                 "eagle_fp8_tp4_cutedsl": {"vllm": V, "flags": base + ["--linear-backend", "flashinfer_cutedsl"]},
                 "eagle_fp8_tp4_fp8kv": {"vllm": V, "flags": base + ["--kv-cache-dtype", "fp8_e4m3"]},
                 "mtp3_fp8_tp4_control": {"vllm": V, "flags": [fp8, "--tensor-parallel-size", "4", "--max-model-len", "106496",
                                          "--max-num-seqs", "16", "--kv-cache-dtype", "fp8_e4m3", "--attention-backend",
                                          "FLASHINFER", "--speculative-config",
                                          json.dumps({"method": "mtp", "num_speculative_tokens": 3}),
                                          "--enable-prefix-caching", "--disable-custom-all-reduce"]}}
        (T / "eagle_flags.json").write_text(json.dumps(flags, indent=1))
        rc, txt = sh("flagcheck", f"cd /tmp && VLLM_USE_V2_MODEL_RUNNER=1 {VPY} {T}/e21_flagcheck.py {T}/eagle_flags.json {V}",
                     env=E, ok_rc=(0, 1, 2))
        m = re.search(r"^FLAGCHECK (.*)$", txt, re.M)
        R["facts"]["flagcheck"] = json.loads(m.group(1)) if m else None
        R["facts"]["flagcheck_lines"] = [l[:700] for l in txt.splitlines() if l.startswith("{")][:8]
        # 5. plugin registration: entry points, SpeculativeConfig method values, attention backend names
        probe = (
            "import importlib.metadata as m, json, typing\n"
            "out = {'entry_points': {g: [str(e) for e in m.entry_points(group=g)] for g in ('vllm.general_plugins', 'vllm.platform_plugins')}}\n"
            "try:\n from vllm.config import SpeculativeConfig\n hints = typing.get_type_hints(SpeculativeConfig)\n"
            " out['spec_method_type'] = str(hints.get('method'))[:1500]\n"
            " out['spec_fields'] = sorted(getattr(SpeculativeConfig, '__dataclass_fields__', {}))\n"
            "except Exception as e:\n out['spec_error'] = repr(e)[:300]\n"
            "try:\n from vllm.v1.attention.backends.registry import AttentionBackendEnum as A\n out['attn_backends'] = [x.name for x in A]\n"
            "except Exception as e:\n out['attn_error'] = repr(e)[:300]\n"
            "print('PLUGIN ' + json.dumps(out))\n")
        (T / "probe.py").write_text(probe)
        rc, txt = sh("plugin", f"cd /tmp && {VPY} {T}/probe.py", env=E, ok_rc=(0, 1))
        m = re.search(r"^PLUGIN (.*)$", txt, re.M)
        R["facts"]["plugin"] = json.loads(m.group(1)) if m else None
        sp = json.dumps(R["facts"]["plugin"] or {})
        R["facts"]["eagle_method_visible"] = "eagle_harness" in sp
        R["facts"]["custom_attention_visible"] = "CUSTOM" in sp
        # 6. compiled-library archs (sm_89, or sm_80 cubins / PTX that run on sm_89) and max GLIBC
        sh("cuobjdump_get", "python3 -m pip download --no-deps -q -d /tmp/cuobj nvidia-cuda-cuobjdump==13.2.51 && "
                            "cd /tmp/cuobj && python3 -c \"import zipfile,glob; zipfile.ZipFile(glob.glob('*.whl')[0]).extractall('x')\" && "
                            "chmod +x x/nvidia/cu13/bin/cuobjdump && x/nvidia/cu13/bin/cuobjdump --version | tail -1", ok_rc=(0, 1))
        cuo = "/tmp/cuobj/x/nvidia/cu13/bin/cuobjdump"
        site = subprocess.run([VPY, "-c", "import sysconfig; print(sysconfig.get_paths()['purelib'])"], capture_output=True, text=True).stdout.strip()
        R["facts"]["site"] = site
        sos = [p for d in ("vllm", "eagle_harness", "flashinfer") for p in Path(site, d).rglob("*.so*") if p.is_file()] if site else []
        archs = {}
        for p in sos:
            r = subprocess.run([cuo, "--list-elf", str(p)], capture_output=True, text=True)
            e = sorted(set(re.findall(r"sm_(\d+)", r.stdout)))
            r2 = subprocess.run([cuo, "--list-ptx", str(p)], capture_output=True, text=True)
            x = sorted(set(re.findall(r"sm_(\d+)", r2.stdout)))
            if e or x:
                archs[str(p.relative_to(site))] = {"elf": e, "ptx": x}
        R["facts"]["archs"] = archs
        R["facts"]["archs_runnable_on_sm89"] = {k: any(a in ("80", "86", "89") for a in v["elf"]) or any(int(a) <= 89 for a in v["ptx"])
                                                for k, v in archs.items()}
        rc, txt = sh("glibc", f"python3 -I {PAY}/glibc235_fix.py --dry-run " + " ".join(str(p) for p in sos[:400]), ok_rc=(0, 2))
        m = re.search(r"^GLIBCFIX (.*)$", txt, re.M)
        g = json.loads(m.group(1)) if m else {}
        R["facts"]["glibc_over_2_35"] = {k: v.get("high_symbols") for k, v in (g.get("files") or {}).items() if v.get("high")}
        # 7. arch gates and backend code in the sources (informational, for the GPU smoke plan)
        sh("source_gates", f"cd {site} && grep -rn -E 'get_device_capability|is_device_capability|compute_capability|sm_?(80|89|90|100|103)\\b' "
                           f"eagle_harness | head -60; echo ----; grep -rln -E 'flashinfer_cutedsl|CUTEDSL|cutedsl' vllm | head -20; "
                           f"echo ----; grep -rn -E 'class .*Custom.*Backend|CUSTOM *=|\"CUSTOM\"' vllm | head -20", ok_rc=(0, 1, 2))
    # 8. payload as ONE .bin (Kaggle unpacks .zip/.tar on Model or dataset creation and checks every inner name)
    if not R["errors"]:
        files = sorted(p for p in PAY.rglob("*") if p.is_file())
        out = W / "eagle_offline_tar.bin"
        with tarfile.open(out, "w", format=tarfile.PAX_FORMAT) as t:
            for p in files:
                t.add(p, "eagle_offline/" + str(p.relative_to(PAY)), recursive=False)
        with tarfile.open(out) as t:
            got = {m.name: m.size for m in t.getmembers() if m.isfile()}
        want = {"eagle_offline/" + str(p.relative_to(PAY)): p.stat().st_size for p in files}
        h = hashlib.sha256()
        with open(out, "rb") as f:
            for b in iter(lambda: f.read(8 << 20), b""):
                h.update(b)
        R["payload"] = {"file": out.name, "bytes": out.stat().st_size, "sha256": h.hexdigest(), "members": len(got),
                        "mismatch": sorted(n for n in set(want) | set(got) if want.get(n) != got.get(n))[:20]}
        if R["payload"]["mismatch"]:
            fail("tar member mismatch")
        if R["payload"]["bytes"] > 19e9:
            fail("payload too big")
except Exception as e:
    import traceback
    fail(f"fatal: {e!r}")
    R["traceback"] = traceback.format_exc()[-3000:]
finally:
    with tarfile.open(W / "p6a_logs_tar.bin", "w") as t:
        for p in sorted(LOGS.glob("*")):
            t.add(p, "p6a_logs/" + p.name, recursive=False)
    bad = sorted(str(p.relative_to(W)) for p in W.rglob("*") if p.is_dir() or (p.is_file() and (
        len(p.relative_to(W).parts) > 1 or re.search(r"\.(zip|tar|gz|tgz|bz2|xz|7z)$", p.name) or re.search(r"[^A-Za-z0-9._-]", p.name))))
    R["bad_output_names"] = bad
    if bad:
        fail(f"output not flat / safe: {bad[:10]}")
    f = R["facts"]
    R["feasibility"] = {"offline_install": "offline_install" in R["steps"] and R["steps"]["offline_install"]["rc"] == 0,
                        "import": f.get("import_ok"), "flagcheck": f.get("flagcheck"), "eagle_method_visible": f.get("eagle_method_visible"),
                        "custom_attention_visible": f.get("custom_attention_visible"),
                        "libs_not_runnable_on_sm89": sorted(k for k, v in (f.get("archs_runnable_on_sm89") or {}).items() if not v),
                        "glibc_over_2_35": f.get("glibc_over_2_35")}
    R["all_ok"] = not R["errors"]
    R["sec"] = round(time.time() - T0)
    (W / "p6a_report.json").write_text(json.dumps(R, indent=1))
    print(json.dumps({"feasibility": R["feasibility"], "versions": f.get("versions"), "wheels": f.get("wheels"),
                      "payload": R.get("payload"), "errors": R["errors"]}, default=str)[:8000], flush=True)
    print("ALL OK" if R["all_ok"] else "NOT OK", "|", R["sec"], "s", flush=True)
